### A. Model Training - Binary SALARY Classification (with proper NaN handling)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, matthews_corrcoef, precision_score, recall_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ====================== LOAD DATA ======================
df = pd.read_parquet("../../data/merged_data/merged.parquet")
print("Original data shape:", df.shape)

# ====================== CREATE BINARY TARGET ======================
target_original = 'salary'
threshold = df[target_original].median()
print(f"Median SALARY threshold used: {threshold:.2f}")

df['SALARY_BINARY'] = (df[target_original] > threshold).astype(int)
target_col = 'SALARY_BINARY'

print(f"\nBinary Target distribution:\n", df[target_col].value_counts(normalize=True))

# ====================== PREPARE FEATURES & TARGET ======================
X = df.drop(columns=[target_original, target_col])
y = df[target_col]

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

print(f"Features shape before cleaning: {X.shape}")

# ====================== HANDLE NaNs WITH IMPUTATION ======================
# Use median imputation (robust to outliers)
imputer = SimpleImputer(strategy='median')

# We'll put imputation inside a pipeline for Logistic Regression
print(f"Total NaNs in features: {X.isna().sum().sum()}")

# ====================== TRAIN-TEST SPLIT ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# ====================== TRAIN MODELS WITH PIPELINES ======================

# Model 1: Logistic Regression with Imputation Pipeline
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000,
        n_jobs=-1
    ))
])

lr_pipeline.fit(X_train, y_train)
print("✓ Logistic Regression (with imputation) trained")

# Model 2: XGBoost (can handle NaNs natively, but we impute for consistency)
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum() if y_train.sum() > 0 else 1,
    random_state=42,
    eval_metric='auc',
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    n_jobs=-1,
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)   # XGBoost can train directly even with NaNs
print("✓ XGBoost trained")

# ====================== EVALUATION ======================
def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    return {
        'Model': name,
        'ROC-AUC': round(roc_auc_score(y_test, y_pred_proba), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4)
    }

lr_metrics = evaluate_model(lr_pipeline, X_test, y_test, 'Logistic Regression')
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

comparison_df = pd.DataFrame([lr_metrics, xgb_metrics])
comparison_df = comparison_df.set_index('Model')

print("\n" + "="*75)
print("MODEL PERFORMANCE COMPARISON ON TEST SET (Binary SALARY)")
print("="*75)
display(comparison_df)

Original data shape: (250000, 18)
Median SALARY threshold used: 143453.00

Binary Target distribution:
 SALARY_BINARY
0    0.500012
1    0.499988
Name: proportion, dtype: float64
Features shape before cleaning: (250000, 10)
Total NaNs in features: 175455
✓ Logistic Regression (with imputation) trained
✓ XGBoost trained

MODEL PERFORMANCE COMPARISON ON TEST SET (Binary SALARY)


,ROC-AUC,MCC,Precision,Recall,F1-Score
Model,,,,,
Logistic Regression,0.8281,0.4956,0.7384,0.7669,0.7524
XGBoost,0.8435,0.5150,0.7524,0.7675,0.7599
